In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# MLflow Lab Model Registry

## Steps
1. Load in Airbnb dataset, and save both training dataset and test dataset as Delta tables
1. Train an MLlib linear regression model using all the listing features and tracking parameters, metrics artifacts and Delta table version to MLflow
1. Register this initial model and move it to staging using MLflow Model Registry
1. Add a new column, **`log_price`** to both our train and test table and update the corresponding Delta tables
1. Train a second MLlib linear regression model, this time using **`log_price`** as our target and training on all features, tracking to MLflow 
1. Compare the performance of the different runs by looking at the underlying data versions for both models
1. Move the better performing model to production in MLflow model registry

In [ ]:
train_delta_path = "/home/jovyan/work/outputs/airbnb/train.delta"
test_delta_path = "/home/jovyan/work/outputs/airbnb/test.delta"




## Load Dataset
Let's load the clean Airbnb dataset in again 
We created it in the previous notebook, it should exists in `/home/jovyan/work/outputs/airbnb/clean_data`

In [ ]:
file_path = f"/home/jovyan/work/outputs/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)

train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

##  Step 1. Creating Delta Tables
We will create a Delta table for each of the datasets

In [ ]:
# In case paths already exists

train_df.write.mode("overwrite").format("delta").save(train_delta_path)
test_df.write.mode("overwrite").format("delta").save(test_delta_path)

Let's read the first version of each delta table

In [ ]:
data_version = 0
train_delta = spark.read.format("delta").option("versionAsOf", data_version).load(train_delta_path)  
test_delta = spark.read.format("delta").option("versionAsOf", data_version).load(test_delta_path)

### Review Delta Table History
All the transactions for this table are stored within this table including the initial set of insertions, update, delete, merge, and inserts.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY delta.`{train_delta_path}`"))

## Step 2. Log Initial Run to MLflow
Let's first log a run to MLflow where we use all features. We use the same approach with RFormula as before. This time however, let's also log both the version of our data and the data path to MLflow.

In [ ]:
import mlflow
import mlflow.spark
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import RFormula

with mlflow.start_run(run_name="lr_model") as run:
    # Log parameters
    mlflow.log_param("label", "price-all-features")
    mlflow.log_param("data_version", data_version)
    mlflow.log_param("data_path", train_delta_path)    

    # Create pipeline
    r_formula = RFormula(formula="price ~ .", featuresCol="features", labelCol="price", handleInvalid="skip")
    lr = LinearRegression(labelCol="price", featuresCol="features")
    pipeline = Pipeline(stages = [r_formula, lr])
    model = pipeline.fit(train_delta)

    # Log pipeline
    mlflow.spark.log_model(model, "model")

    # Create predictions and metrics
    pred_df = model.transform(test_delta)
    regression_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")
    rmse = regression_evaluator.setMetricName("rmse").evaluate(pred_df)
    r2 = regression_evaluator.setMetricName("r2").evaluate(pred_df)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    run_id = run.info.run_id

## Step 3. Register Model and Move to Staging Using MLflow Model Registry
We are happy with the performance of the above model and want to move it to staging. Let's create the model and register it to the MLflow model registry.

In [ ]:
model_uri = f"runs:/{run_id}/model"

suffix = "scc"
model_name = f"mllib-lr_{suffix}"
print(f"Model Name: {model_name}\n")

model_details = mlflow.register_model(model_uri=model_uri, name=model_name)

Transition model to staging.

In [ ]:
from mlflow.tracking.client import MlflowClient

client = MlflowClient()

client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Staging"
)

In [ ]:
# Define a utility method to wait until the model is ready
def wait_for_model(model_name, version, stage="None", status="READY", timeout=300):
    import time

    last_stage = "unknown"
    last_status = "unknown"

    for i in range(timeout):
        model_version_details = client.get_model_version(name=model_name, version=version)
        last_stage = str(model_version_details.current_stage)
        last_status = str(model_version_details.status)
        if last_status == str(status) and last_stage == str(stage):
            return

        time.sleep(1)

    raise Exception(f"The model {model_name} v{version} was not {status} after {timeout} seconds: {last_status}/{last_stage}")

In [ ]:
# Force our notebook to block until the model is ready
wait_for_model(model_name, 1, stage="Staging")

Add a model description using <a href="https://mlflow.org/docs/latest/python_api/mlflow.client.html#mlflow.client.MlflowClient.update_registered_model" target="_blank">update_registered_model</a>.

In [ ]:
client.update_registered_model(
    name=model_details.name,
    description="This model forecasts Airbnb housing list prices based on various listing inputs."
)

In [ ]:
wait_for_model(model_details.name, 1, stage="Staging")

##  Step 4. Feature Engineering: Evolve Data Schema
We now want to do some feature engineering with the aim of improving model performance; we can use Delta Lake to track older versions of the dataset. 
We will add **`log_price`** as a new column and update our Delta table with it.

In [ ]:
from pyspark.sql.functions import col, log, exp

# Create a new log_price column for both train and test datasets
train_new = train_delta.withColumn("log_price", log(col("price")))
test_new = test_delta.withColumn("log_price", log(col("price")))

Save the updated DataFrames to **`train_delta_path`** and **`test_delta_path`**, respectively, passing the **`mergeSchema`** option to safely evolve its schema. 
Take a look at this <a href="https://databricks.com/blog/2019/09/24/diving-into-delta-lake-schema-enforcement-evolution.html" target="_blank">blog</a> on Delta Lake for more information about **`mergeSchema`**.

In [ ]:
train_new.write.option("mergeSchema", "true").format("delta").mode("overwrite").save(train_delta_path)
test_new.write.option("mergeSchema", "true").format("delta").mode("overwrite").save(test_delta_path)

Reviewing delta history

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY delta.`{train_delta_path}`"))

In [ ]:
data_version = 1
train_delta_new = spark.read.format("delta").option("versionAsOf", data_version).load(train_delta_path)  
test_delta_new = spark.read.format("delta").option("versionAsOf", data_version).load(test_delta_path)

## Step 5. Use **`log_price`** as Target and Track Run with MLflow
Retrain the model on the updated data and compare its performance to the original, logging results to MLflow.

In [ ]:
with mlflow.start_run(run_name="lr_log_model") as run:
    # Log parameters
    mlflow.log_param("label", "log-price")
    mlflow.log_param("data_version", data_version)
    mlflow.log_param("data_path", train_delta_path)    

    # Create pipeline
    r_formula = RFormula(formula="log_price ~ . - price", featuresCol="features", labelCol="log_price", handleInvalid="skip")  
    lr = LinearRegression(labelCol="log_price", predictionCol="log_prediction")
    pipeline = Pipeline(stages = [r_formula, lr])
    pipeline_model = pipeline.fit(train_delta_new)

    # Log model and update the registered model
    mlflow.spark.log_model(
        spark_model=pipeline_model,
        artifact_path="log-model",
        registered_model_name=model_name
    )  

    # Create predictions and metrics
    pred_df = pipeline_model.transform(test_delta)
    exp_df = pred_df.withColumn("prediction", exp(col("log_prediction")))
    rmse = regression_evaluator.setMetricName("rmse").evaluate(exp_df)
    r2 = regression_evaluator.setMetricName("r2").evaluate(exp_df)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)  

    run_id = run.info.run_id

## Step 6. Compare Performance Across Runs by Looking at Delta Table Versions 
Use MLflow's <a href="https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.search_runs" target="_blank">**`mlflow.search_runs`**</a> API to identify runs according to the version of data the run was trained on. Let's compare our runs according to our data versions.
Filter based on **`params.data_path`** and **`params.data_version`**.

In [ ]:
data_version = 0

mlflow.search_runs(filter_string=f"params.data_path='{train_delta_path}' and params.data_version='{data_version}'")

In [ ]:
data_version = 1

mlflow.search_runs(filter_string=f"params.data_path='{train_delta_path}' and params.data_version='{data_version}'")

## Step 7. Move the Best Performing Model to Production Using MLflow Model Registry
Get the most recent model version and move it to production.

In [ ]:
model_version_infos = client.search_model_versions(f"name = '{model_name}'")
new_model_version = max([model_version_info.version for model_version_info in model_version_infos])

In [ ]:
client.update_model_version(
    name=model_name,
    version=new_model_version,
    description="This model version was built using a MLlib Linear Regression model with all features and log_price as predictor."
)

In [ ]:
model_version_details = client.get_model_version(name=model_name, version=new_model_version)
model_version_details.status

In [ ]:
wait_for_model(model_name, new_model_version)

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=new_model_version,
    stage="Production"
)

In [ ]:
wait_for_model(model_name, new_model_version, "Production")

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Archived"
)

In [ ]:
wait_for_model(model_name, 1, "Archived")

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=2,
    stage="Archived"
)

In [ ]:
wait_for_model(model_name, 2, "Archived")

In [ ]:
client.delete_registered_model(model_name)